In [1]:
import os
import gc
from scipy.ndimage import gaussian_filter
import pyclesperanto_prototype as cle
# initialize GPU
device = cle.select_device("GTX")
print("Used GPU: ", device)


import numpy as np

from os.path import getsize
import napari
from skimage.io import imread

filename =  'Image_001_001.raw'
previewFilename = 'ChanC_Preview.tif'
MAXCHUNKSIZE = 1024*288*2*1500
class thorlabsFile():
   
    def __init__(self,folder) -> None:

        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)

        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width),dtype=np.uint16)
        self.app = napari.Viewer()
        #self.app.add_image(self.array)

    def loadFile(self,folder):

        try:
            self.r.close()
        except:
            pass
        
        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)
        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width),dtype=np.uint16)

        l = self.app.layers['Image']
        l.data = self.array
        
    def getImage(self,n):

        offset = n*self.frameSize
        self.r.seek(offset)
        st = self.r.read(self.frameSize)
        nparray = np.frombuffer(st,dtype = np.uint16).reshape((1,self.height,self.width))
        
        return nparray

    def loadWholeStack(self,start=0,end=-1,step=1):

        
        if end == -1:
            end = nFrames
        totalFrames = end-start
        totalFramesSize = totalFrames*self.frameSize

        if totalFramesSize<=MAXCHUNKSIZE:

            offset = start*self.frameSize
            self.r.seek(offset)
            st = self.r.read(totalFramesSize)
            stack = np.frombuffer(st,dtype = np.uint16).reshape((totalFrames,self.height,self.width))

            stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)

   
            stack3 = [stack]
            
        else:
            chunksizeFrames = MAXCHUNKSIZE//(self.frameSize)  #number of frames in a chunk
            nchunks = totalFrames//chunksizeFrames
            remainderFrames = totalFrames%chunksizeFrames
            stack3 = []
            for i in range(nchunks):
                offset = (start+i*chunksizeFrames)*self.frameSize
                self.r.seek(offset)
                st = self.r.read(self.frameSize*chunksizeFrames)
                stack = np.frombuffer(st,dtype = np.uint16).reshape((chunksizeFrames,self.height,self.width))

                stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                stack3.append(stack)
            if remainderFrames != 0:
                offset = (start+nchunks*chunksizeFrames)*self.frameSize
                self.r.seek(offset)
                st = self.r.read(self.frameSize*remainderFrames)
                stack = np.frombuffer(st,dtype = np.uint16).reshape((remainderFrames,self.height,self.width))

                stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                stack3.append(stack)

        return stack3

    def loadNextNFrames(self,n):
        newStacks = self.loadWholeStack(start=self.currentLastFrame,end = self.currentLastFrame+n)
        self.array= np.vstack([self.array, *newStacks])
        del newStacks
        try:
            l = self.app.layers['Image']
            l.data = self.array
        except:
            self.app.add_image(self.array)

        self.currentLastFrame = self.array.shape[0]

    def loadUpToFrameN(self,n):
        
        if (n<self.currentLastFrame) | (n>self.nFrames):
            return
        newStack = self.loadWholeStack(start=self.currentLastFrame,end = n)
        self.array= np.vstack([self.array, *newStack])

        if self.currentLastFrame == 0:
                self.app.add_image(self.array)
        else:
            l = self.app.layers['Image']
            l.data = self.array
        self.currentLastFrame = self.array.shape[0]




ModuleNotFoundError: No module named 'pyclesperanto_prototype'

In [4]:
fn = '../sampleImage/'

try:
    tb.r.close()
    del tb.array
    tb.loadFile(fn)
    gc.collect()
except:
    print('nope')
    tb = thorlabsFile(fn)

In [5]:

tb.loadNextNFrames(1500)